# 01 — Préparation des données

Split stratifié train / val / test du dataset d'images thermiques de moteurs à induction (6 classes : HFL, HML, HNL, CFL, CML, CNL).

**Structure attendue en entrée** (`data/raw/`) :
```
data/raw/HFL/*.jpg
data/raw/HML/*.jpg
data/raw/HNL/*.jpg
data/raw/CFL/*.jpg
data/raw/CML/*.jpg
data/raw/CNL/*.jpg
```


In [ ]:
import csv
import random
from pathlib import Path
from collections import defaultdict

## Configuration

In [ ]:
# Effectifs attendus (pour validation / avertissement si le dataset livré diffère)
EXPECTED_COUNTS = {
    "HFL": 112,
    "HML": 82,
    "HNL": 90,
    "CFL": 49,
    "CML": 79,
    "CNL": 76,
}

CLASSES = list(EXPECTED_COUNTS.keys())
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# Paramètres modifiables
DATA_DIR = Path("../data/raw")
OUT_DIR = Path("../data/processed")
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
SEED = 42

## Fonctions

In [ ]:
def collect_images(data_dir: Path) -> dict:
    """Retourne {classe: [liste de chemins d'images]}."""
    images_by_class = defaultdict(list)
    for cls in CLASSES:
        cls_dir = data_dir / cls
        if not cls_dir.exists():
            print(f"  ATTENTION : dossier manquant pour la classe '{cls}' ({cls_dir})")
            continue
        for f in sorted(cls_dir.iterdir()):
            if f.suffix.lower() in IMG_EXTENSIONS:
                images_by_class[cls].append(f)
    return images_by_class

In [ ]:
def stratified_split(images_by_class: dict, train_frac=0.70, val_frac=0.15, seed=42):
    """
    Split stratifié par classe. Le reste après train+val va au test.
    Chaque classe est mélangée indépendamment (seed fixe pour reproductibilité).
    """
    rng = random.Random(seed)
    splits = {"train": [], "val": [], "test": []}

    for cls, files in images_by_class.items():
        files = files.copy()
        rng.shuffle(files)
        n = len(files)
        n_train = round(n * train_frac)
        n_val = round(n * val_frac)
        # garde-fou : au moins 1 image en val/test si la classe est petite
        n_train = min(n_train, n - 2) if n >= 3 else n_train

        train_files = files[:n_train]
        val_files = files[n_train:n_train + n_val]
        test_files = files[n_train + n_val:]

        for f in train_files:
            splits["train"].append((f, cls))
        for f in val_files:
            splits["val"].append((f, cls))
        for f in test_files:
            splits["test"].append((f, cls))

    return splits

In [ ]:
def print_report(images_by_class: dict, splits: dict):
    print("\n--- Effectifs par classe (dataset fourni) ---")
    total = 0
    for cls in CLASSES:
        n = len(images_by_class.get(cls, []))
        total += n
        expected = EXPECTED_COUNTS[cls]
        flag = "" if n == expected else f"  (attendu: {expected})"
        print(f"  {cls}: {n}{flag}")
    print(f"  TOTAL: {total}")

    print("\n--- Répartition train / val / test ---")
    for split_name in ["train", "val", "test"]:
        counts = defaultdict(int)
        for _, cls in splits[split_name]:
            counts[cls] += 1
        n_total = sum(counts.values())
        print(f"  {split_name} ({n_total} images):")
        for cls in CLASSES:
            print(f"      {cls}: {counts.get(cls, 0)}")

In [ ]:
def save_splits_csv(splits: dict, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["filepath", "class", "split"])
        for split_name in ["train", "val", "test"]:
            for filepath, cls in splits[split_name]:
                writer.writerow([str(filepath), cls, split_name])
    print(f"\nSplits sauvegardés dans : {out_path}")

## Exécution

In [ ]:
print(f"Lecture des images depuis : {DATA_DIR.resolve()}")
images_by_class = collect_images(DATA_DIR)

if not images_by_class:
    print("Aucune image trouvée. Vérifie que data/raw/<CLASSE>/ contient bien les images.")
else:
    splits = stratified_split(images_by_class, TRAIN_FRAC, VAL_FRAC, SEED)
    print_report(images_by_class, splits)
    save_splits_csv(splits, OUT_DIR / "splits.csv")